# AIOps CA-2: Spambase Spam Email Classification

**Goal:** classify email feature vectors as spam (1) or legitimate/ham (0).

Run this notebook in **Google Colab**, not VS Code.

Dataset: `spambase_csv.csv` (4,601 rows, 57 numerical features, target column `class`).

Workflow: load data → inspect → split → train → evaluate → fit final pipeline → save `model.joblib`.


## Step 1 — Upload the CSV file to Colab
Open [Google Colab](https://colab.research.google.com/), choose **New notebook**, then use the folder icon on the left → **Upload to session storage** → select `spambase_csv.csv`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

print("Libraries imported successfully")


In [ ]:
df = pd.read_csv("/content/spambase_csv.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values:", df.isnull().sum().sum())
print("\\nClass distribution:")
print(df["class"].value_counts().sort_index().rename(index={0: "Ham (0)", 1: "Spam (1)"}))


In [ ]:
# Separate input features and target
X = df.drop(columns=["class"])
y = df["class"]

print("Number of features:", X.shape[1])
print("Target classes:", sorted(y.unique()))


## Step 2 — Split into training and testing sets
We reserve 20% for testing. `stratify=y` preserves the class balance in both sets. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


## Step 3 — Train Logistic Regression
The pipeline scales the features first and then trains Logistic Regression. Keeping both steps in one pipeline ensures the same preprocessing is used during prediction.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

model.fit(X_train, y_train)
print("Model training completed.")


## Step 4 — Evaluate the model on unseen test data

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")
print("\\nConfusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_test, y_pred))
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))


### What these metrics mean
- **Accuracy:** fraction of all emails classified correctly.
- **Precision:** of the emails predicted as spam, how many really are spam.
- **Recall:** of all actual spam emails, how many the model catches.
- **F1-score:** balance between precision and recall.
- **Confusion matrix:** counts correct and incorrect predictions by class.


## Step 5 — Train the final model on all available data and save it
After evaluating on the held-out test set, we fit a deployment model on the full dataset. This saved pipeline includes the scaler and classifier.

In [ ]:
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])
final_model.fit(X, y)

joblib.dump(final_model, "model.joblib")
print("Saved model.joblib")


## Step 6 — Download `model.joblib`
Run the cell below. Your browser will download the file. Later, copy it into the `spambase-aiops-ca2` project folder in VS Code, replacing the starter model.

In [ ]:
from google.colab import files
files.download("model.joblib")


## Step 7 — Quick local prediction sanity check

In [ ]:
sample = X.iloc[[0]]
prediction = int(final_model.predict(sample)[0])
label = "Spam" if prediction == 1 else "Ham (not spam)"
print("Prediction for first dataset row:", prediction, "-", label)
